In [0]:
silver_df = spark.read.table('it_support_tickets.bronze.bronze_table')

In [0]:
display(silver_df)

In [0]:
from pyspark.sql.functions import *

silver_df = silver_df.withColumn(
    "ticket_id", upper(trim(col("ticket_id")))
).withColumn(
    "employee_id", upper(trim(col("employee_id")))
).withColumn(
    "support_category", upper(trim(col("support_category")))
).withColumn(
    "priority", upper(trim(col("priority")))
).withColumn(
    "ticket_status",
    regexp_replace(upper(trim(col("ticket_status"))), " ", "_")
).withColumn(
    "location", initcap(trim(col("location")))
).withColumn(
    "support_engineer", trim(col("support_engineer"))
)

In [0]:
silver_df = silver_df.withColumn(
    "support_category",
    when(col("support_category") == "HARDWRE", "HARDWARE")
    .when(col("support_category") == "NETWORKING", "NETWORK")
    .otherwise(col("support_category"))
)
silver_df = silver_df.withColumn(
    "priority",
    when(col("priority") == "URGENT", "CRITICAL")
    .otherwise(col("priority"))
)

In [0]:
silver_df = silver_df.withColumn(
    "priority",
    when(col("priority") == "CRITCAL", "CRITICAL")
    .otherwise(col("priority"))
).withColumn(
    "ticket_status",
    when(col("ticket_status") == "IN PROGRESS", "IN_PROGRESS" )
    .when(col("ticket_status") == "CLOSED??", "RESOLVED")
    .otherwise(col("ticket_status"))
)

In [0]:
display(silver_df)

In [0]:
silver_df = silver_df.withColumn(
    "total_resolution_cost",
    col("resolution_hours") * col("hourly_cost")
)

In [0]:
display(silver_df)

In [0]:
display(silver_df)

print("Silver record count:", silver_df.count())

silver_df.printSchema()

In [0]:
silver_df.write.mode("overwrite").format('delta').option('mergeSchema','true').saveAsTable("it_support_tickets.silver.silver_table")

In [0]:
%sql
select * from it_support_tickets.silver.silver_table